In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter
from utils import *
from cavity_correction import correct_cavity
from prefilter_correction import correct_prefilter

In [2]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/'
flat_folders = sorted(glob.glob(folder + '*'))

print(flat_folders)
flat_folder = flat_folders[-7]
flat_files = sorted(glob.glob(flat_folder + '/*.fits'))

['/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-04-03', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-04-06', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-04-15', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-10-11', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-03-30', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-09-26', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-10-16', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-10-27', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-12-02', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-01-19', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-03-10', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-09-15', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-09-23', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2026-03-10', '/home/ulyanov/data/solo/phi/flat

In [3]:
flat_files = sorted(glob.glob('../process/temp/*.fits'))
print(flat_files)

['../process/temp/phi-fdt-cavity_20250310T080009_V202608171731C_0563100100.fits', '../process/temp/phi-fdt-flat_20250310T080009_V202608171731C_0563100100.fits', '../process/temp/phi-fdt-ghost_20250310T080009_V202608171731C_0563100100.fits']


In [4]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'

cavity_file, flat_file, ghost_file = flat_files

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(cavity_file) as hdul:
    cavity = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data
    header = hdul[0].header

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

ghost = demodulate(ghost, header)
flat_ = demodulate(flat, header)
flat_ -= np.mean(flat_, axis=(-2,-1), keepdims=True)

In [5]:
plt.figure(figsize=(10,10))
plt.imshow(cavity, 'bwr', vmin=-5e-2, vmax=5e-2)
plt.tight_layout()

In [6]:
plt.figure(figsize=(10,10))
plt.imshow(flat_[1], 'gray', vmin=-5e-3, vmax=5e-3)
plt.tight_layout()

In [7]:
plt.figure(figsize=(10,10))
plt.imshow(flat[0], 'gray', vmin=0.8, vmax=1.1)
plt.tight_layout()

In [8]:
def calc_ghost_scaling(data, header, sigma=0.043, gamma=0.053, depth=0.66):
    from scipy.signal import convolve

    dx = 0.001
    x = np.arange(-10,10 + dx / 2, dx)
    f = 1 - depth * np.exp(-x ** 2 / 2 / sigma ** 2)
    g = 1 / (1 + (x / gamma) ** 2)
    q = 2 * (1 - convolve(f, g ** 2, mode='same') / convolve(f, g, mode='same'))

    shift = get_wv_shift(data, header)

    Q = interpolate(q.reshape(-1,1,1), x, wv.reshape(-1,1,1) - np.expand_dims(shift, axis=0))
    return np.insert(np.nan_to_num(Q, nan=1), cpos, np.ones_like(shift), axis=0)

In [9]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/'
#folder = '/home/ulyanov/data/solo/phi/test/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T080009_V202503131733C_0563100100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T080608_V202503131733C_0563100125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T081208_V202503131835C_0563100150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T081808_V202503131935C_0563100175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T082408_V202503131935C_0563100200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T083008_V202503132033C_0563100225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T083608_V202503141634C_0563100250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025

In [10]:
with fits.open(files[0]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

xr, yr = reflection_point_predict(header)
wv = read_wavelengths(header)
cpos = header['CONTPOS'] - 1
print(cpos)

nx, ny = data.shape[-2:]

data = data.reshape(6,4,nx,ny)
data -= crop(dark, header) * 0.4
data = correct_prefilter(data, header, prefilter_file)
data /= crop(flat, header)

data = correct_cavity(data, header, cavity)
q = calc_ghost_scaling(data, header)

5


In [11]:
plt.figure(figsize=(10,10))
plt.imshow(data[0,0])
plt.tight_layout()

In [12]:
wv_shift = get_wv_shift(data, header)

In [13]:
plt.figure(figsize=(10,10))
plt.imshow(wv_shift, 'bwr', vmin=-5e-2, vmax=5e-2)
plt.tight_layout()

In [16]:
i = cpos

temp = data[i].copy()
temp = realign(temp)
temp = demodulate(temp, header)

reflection = reflect(gaussian_filter(temp[0], 8), xr, yr)

In [17]:
a, b = np.nanpercentile(temp[0], 0.1), np.nanpercentile(temp[0], 99.9)

j = 1

plt.figure(figsize=(10,10))
plt.imshow(temp[j] - crop(ghost, header)[j] * reflection * q[i], 'gray', vmin=-1e-3 * (b - a), vmax=1e-3 * (b - a), origin='lower')
plt.tight_layout()